# Getting Started with basic RL-Framework: Environment
In RL, an environment can be consider as a mapping from action space $\mathcal{A}$ to state space $\mathcal{S}$. In a stochastic setting, an environment is characterized by the state transition probability:

$$
p(s_{t+1} = s'|s_t = s, a_t =a)
$$

which is a (conditional) probability that the next state $s_{t+1} = s'$ given the curent state $s_t=s$ and action $a_t=a$.

Taking an action $a_t$ in the current state is $s_t$ will return a $\bold{scalar}$ signal called $\bold{reward}$:

$$
r(s_t, a_t) \in R
$$

We first take an defined environemnt in gymnasium called "Cart-pole"

In [3]:
import gymnasium as gym
print("Imported gymnasimum. Version: ", gym.__version__)
cartpole_env = gym.make("CartPole-v1", render_mode = None)
print("Environment created: ", cartpole_env.spec.id)

Imported gymnasimum. Version:  1.4.0
Environment created:  CartPole-v1


We check some basic information of this env include the dimension of the state/observation space and dimension of the action space. In this env, they are:

$$dim\mathcal(S) = 4 \\
dim(\mathcal{A}) = 2
$$

The state space $\mathcal{S}$ encode the cart and pole's position and velocity while the action space $\mathcal{A} = \{0,1\}$ represent the PUSH LEFT or PUSH RIGHT with constant force F = 10N

In [3]:
ob_dim = cartpole_env.observation_space.shape[0]
ac_dim = cartpole_env.observation_space.shape[0]

print("Observation dim: ", ob_dim)
print("Action dim: ", ob_dim)

Observation dim:  4
Action dim:  4


An important method of a RL env is reset which reset the env'state to the initial state

In [4]:
obs,__ = cartpole_env.reset() # reset env
print(obs) # print out the initial state

[ 0.02451947  0.02168865 -0.04202878  0.03450857]


We can sample a random action from the env class

In [5]:
action = cartpole_env.action_space.sample() # take a random action
print("Sampled action: ", action)

obs, reward, done, truncated, info = cartpole_env.step(action) # apllying action to the env


Sampled action:  0


In RL, we need to sample rollouts to collect the data and update the policy, a rollout return a trajectory sample which is a sequence of state, action and reward ${s_t,a_t,r_t}$

In [11]:
from typing import Dict
import numpy as np
def sample_trajectory (env: gym.Env, max_total_steps: int) -> Dict:
    # reset env
    ob,_ = env.reset()
    obs, acts, rewards = [], [], []
    num_steps = 0
    done = False
    while not done:
        # sample an action
        action = env.action_space.sample()
        ob, r, terminated, truncated, info = env.step(action)

        # append to the list
        obs.append(ob)
        acts.append(action)
        rewards.append(r)

        # increase the step
        num_steps += 1

        done = terminated or num_steps >= max_total_steps

        if done:
            break

    return {
        "observations": obs,
        "actions": acts,
        "rewards": rewards
    } 

rollout = sample_trajectory(cartpole_env, 200)  

print("Number of step: ", len(rollout["rewards"]))
print("Rewards: ", rollout["rewards"])
print("Actions: ", rollout["actions"])
print("Observation shape: ", np.shape(rollout["observations"]))
    

Number of step:  10
Rewards:  [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
Actions:  [np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1)]
Observation shape:  (10, 4)


In RL, we take care about a metric called Return:
$$
G(t) = \sum_t^H{r(s_t, a_t)}
$$

$G(t)$ is a random scalar as it depends on the sampled trajectory which is a random vector drawn by the trajectory distribution $p_{\pi_\theta}(\tau)$. The goal of RL is to find an optimal policy that maximize the expected return:

$$
\max_{\pi_\theta}{G(t)} = \max_{\pi_\theta}{\sum_t^H{r(s_t, a_t)}}
$$

We implement a function that calculate the return of a trajectory:

In [12]:
def calculate_return (trajectory: Dict) -> np.float32:
    return np.sum(trajectory["rewards"])

R = calculate_return(rollout)
R

np.float64(10.0)

In RL, the return function $G(t) = \sum_t^H{r(s_t, a_t)}$ can be infinitely large, especially if the time horizon is infinite. In this case the algorithm may not be converged. In order to avoid this, we consider discounting the return:

$$
G(t) = \sum_t{\gamma^t r(s_t, a_t)} = r(s_0, a_0) + \gamma r(s_1, a_1) + \gamma^2 r(s_2, a_2) + ...
$$

where $\gamma \in (0,1)$.

With the discounted factor $\gamma \in (0,1) $, the objective function can be ensured to be finite. Using discounted return carries another meaning that we more care about reward in the near future rather than far

$\gamma =\{0.99,0.98\}$ usualy work fine. We consider $\gamma$ as a tuning parameter

In [15]:
gamma = 0.99

def calculate_discounted_return(trajecotry: Dict)-> np.float32:
    rewards = trajecotry["rewards"]
    coeff = np.arange(len(rewards))

    return np.sum(gamma ** coeff * rewards)

Gt = calculate_discounted_return(rollout)
print("Discounted Return: ", Gt)

Discounted Return:  9.561792499119552
